[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Constraints &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the notebook's three constrained tables, the
stations and their year of readings, Kirkenes's two notes and Bjornoya, which has no code, and opens
a connection with `open_database`, which enforces foreign keys. Run it first. The tasks do not
depend on one another, and the last cell closes the connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73, "Bjornoya": 74.50}
CODES = {"Bergen": "BGO", "Oslo": "OSL", "Svalbard": "LYR", "Tromso": "TOS", "Kirkenes": "KKN", "Bjornoya": None}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def open_database(path):
    """A connection that enforces foreign keys, with autocommit=False."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.autocommit = False
    return conn


conn = open_database(DATABASE)
conn.executescript("""
    CREATE TABLE stations (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL UNIQUE,
        code     TEXT UNIQUE,
        latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)
    ) STRICT;
    CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        source     TEXT,
        UNIQUE (station_id, hour)
    ) STRICT;
    CREATE TABLE notes (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id) ON DELETE CASCADE,
        note       TEXT NOT NULL
    ) STRICT;
""")
station_ids = {name: conn.execute("INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)",
                                  (name, CODES[name], latitude)).lastrowid
               for name, latitude in LATITUDES.items()}
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 ((station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.executemany("INSERT INTO notes (station_id, note) VALUES (?, ?)",
                 [(station_ids["Kirkenes"], "Heater checked before winter"), (station_ids["Kirkenes"], "Mast repainted")])
conn.commit()

print("built", DATABASE)


built scratch/stations.db


**1.** A second station called Oslo.


In [2]:
try:
    conn.execute("INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)", ("Oslo", None, 59.9))
except sqlite3.Error as error:
    print(type(error).__name__, error.sqlite_errorname, error.sqlite_errorcode)


IntegrityError SQLITE_CONSTRAINT_UNIQUE 2067


`except sqlite3.Error` catches every sqlite3 error, and the class name shows the refusal was an
`IntegrityError`. `sqlite_errorname` and `sqlite_errorcode` say it was the `UNIQUE` constraint on
`name`.


**2.** A duplicate that changes nothing.


In [3]:
reading = "SELECT id, celsius, source FROM readings WHERE station_id = ? AND hour = ?"
key = (station_ids["Bergen"], "2025-06-21T12:00")
print("before:", conn.execute(reading, key).fetchone())

cursor = conn.execute("""
    INSERT INTO readings (station_id, hour, celsius, source) VALUES (?, ?, ?, ?)
    ON CONFLICT (station_id, hour) DO NOTHING
""", (*key, 17.0, "duplicate"))
conn.commit()
print("rows changed:", cursor.rowcount, "| after:", conn.execute(reading, key).fetchone())


before: (16465, 17.9, None)
rows changed: 0 | after: (16465, 17.9, None)


The reading for that hour was already there, so `DO NOTHING` changed no row, and the stored reading
kept its id, its temperature and its empty source. The new row's own `NOT NULL` and `CHECK`
constraints are still checked before the conflict is found, so a duplicate with a temperature of 99.0
would have been refused by `plausible_celsius`, not skipped.


**3.** A correction that keeps the row's id.


In [4]:
key = (station_ids["Tromso"], "2025-01-01T00:00")
print("before:", conn.execute(reading, key).fetchone())

returned = conn.execute("""
    INSERT INTO readings (station_id, hour, celsius, source) VALUES (?, ?, ?, ?)
    ON CONFLICT (station_id, hour) DO UPDATE SET celsius = excluded.celsius, source = excluded.source
    RETURNING id
""", (*key, -2.4, "correction")).fetchone()
conn.commit()
print("RETURNING:", returned, "| after:", conn.execute(reading, key).fetchone())


before: (4, -6.7, None)
RETURNING: (4,) | after: (4, -2.4, 'correction')


`DO UPDATE` changed the existing row in place, so `RETURNING` gave back the id the reading already
had, and `excluded.celsius` and `excluded.source` were the values the insert tried to add.


**4.** A second connection that enforces foreign keys.


In [5]:
second = open_database(DATABASE)
print("foreign keys:", second.execute("PRAGMA foreign_keys").fetchone()[0])
try:
    second.execute("INSERT INTO notes (station_id, note) VALUES (?, ?)", (99, "A note about nowhere"))
except sqlite3.IntegrityError as error:
    print("refused:", error)
second.close()


foreign keys: 1
refused: FOREIGN KEY constraint failed


`open_database` set the `PRAGMA` before any transaction opened, which is the only time it takes
effect, so the second connection enforced the foreign key from `notes` to `stations` and refused a
note for a station that does not exist. Every connection has to ask, since the setting is not stored
in the file.


**5.** What stops Svalbard being deleted.


In [6]:
try:
    conn.execute("DELETE FROM stations WHERE name = ?", ("Svalbard",))
except sqlite3.IntegrityError as error:
    print("refused:", error)
conn.rollback()

for table in ("readings", "notes"):
    count = conn.execute(f"SELECT COUNT(*) FROM {table} WHERE station_id = ?", (station_ids["Svalbard"],)).fetchone()[0]
    print(f"{table:<9} pointing at Svalbard: {count}")


refused: FOREIGN KEY constraint failed
readings  pointing at Svalbard: 8760
notes     pointing at Svalbard: 0


The 8,760 readings that point at Svalbard refused the delete, since `readings` uses the default,
`NO ACTION`. Svalbard has no notes, and notes would not have stopped it anyway, since they cascade.
The table names in the f-string come from a fixed tuple in the code, never from input.


**6.** `!=` and `IS NOT` beside `NULL`.


In [7]:
with_not_equal = conn.execute("SELECT COUNT(*) FROM stations WHERE code != 'OSL'").fetchone()[0]
with_is_not = conn.execute("SELECT COUNT(*) FROM stations WHERE code IS NOT 'OSL'").fetchone()[0]

print("code != 'OSL':    ", with_not_equal)
print("code IS NOT 'OSL':", with_is_not)


code != 'OSL':     4
code IS NOT 'OSL': 5


Bjornoya has no code. `NULL != 'OSL'` is neither true nor false, so `WHERE` left Bjornoya out of the
first count, while `IS NOT` treats `NULL` as a value, different from `'OSL'`, and counted it.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Constraints](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/10-constraints.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
